# NB5 — Official IqraEval scorer (the TRUE F1)

Our hand-rolled F1 in NB4 (0.364) likely **over-counts False Rejects** because it
penalises prediction-only insertions and connected-speech elisions that the
OFFICIAL scorer ignores. This notebook runs the model on QuranMB.v2 and scores
with the actual IqraEval scripts (`mdd_eval/align_data.py` + `ins_del_cor_sub_analysis.py`),
so we get the real, leaderboard-comparable F1 before doing any further work.

**Official metric (from their code):**
- FR = `cor_nocor` (canonical was correct but model flagged it)
- TR = sub/del/ins matches between annotation and prediction
- FA = `sub_nosub + ins_noins + del_nodel`
- Precision=TR/(TR+FR), Recall=TR/(TR+FA), F1=2PR/(P+R)

**Key fact:** the official truth uses the SAME un-subscripted symbols our model
outputs (`< i nn aa ll ...`), so NO phoneme remapping is needed here.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa pandas
# Official scorer:
!git clone -q https://github.com/Iqra-Eval/interspeech_IqraEval.git || echo 'already cloned'
!ls interspeech_IqraEval/mdd_eval

In [ ]:
import os, numpy as np, torch, pandas as pd

try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
    print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN — need it for the gated GT truth file:", e)

MODEL_DIR = "/kaggle/input/datasets/qasimmaajid/quran-phoneme-model/w2vbert-quran-phoneme/final"   # EDIT
SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| model exists:", os.path.isdir(MODEL_DIR))

In [ ]:
from transformers import AutoModelForCTC, AutoFeatureExtractor, Wav2Vec2CTCTokenizer

model = AutoModelForCTC.from_pretrained(MODEL_DIR).to(DEVICE).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(MODEL_DIR)
id2tok = {i: t for t, i in tokenizer.get_vocab().items()}
blank_id = tokenizer.pad_token_id

def recognise_phonemes(audio_array):
    feats = feature_extractor(np.asarray(audio_array, dtype=np.float32),
                              sampling_rate=SAMPLING_RATE, return_tensors="pt")
    feats = {k: v.to(DEVICE) for k, v in feats.items()}
    with torch.no_grad():
        ids = model(**feats).logits[0].argmax(-1).tolist()
    out, prev = [], None
    for i in ids:
        if i != prev and i != blank_id:
            sym = id2tok.get(int(i), "<unk>")
            if sym not in ("<sil>", "sil", "<pad>", "<unk>"):
                out.append(sym)
        prev = i
    return " ".join(out)

print("model loaded, vocab:", len(id2tok))

In [ ]:
# ---- Run model on all QuranMB.v2 clips -> prediction CSV (ID, Prediction) ----
from datasets import load_dataset, Audio

qmb = load_dataset("IqraEval/QuranMB.v2", split="test")
qmb = qmb.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
print(qmb)

ids, preds = [], []
for k, ex in enumerate(qmb):
    ids.append(ex["ID"])
    preds.append(recognise_phonemes(ex["audio"]["array"]))
    if (k+1) % 200 == 0:
        print(f"  {k+1}/{len(qmb)}")

pred_df = pd.DataFrame({"ID": ids, "Prediction": preds})
pred_df.to_csv("pred.csv", index=False)
print("wrote pred.csv:", pred_df.shape)
print(pred_df.head(3).to_string())

In [ ]:
# ---- Build the TRUTH csv from the gated GT (ID, Reference_phn, Annotation_phn) ----
# The official align_data.py reads columns Reference_phn + Annotation_phn.
gt = load_dataset("IqraEval/IqraEval_Test_GT", split="test")
print("GT columns:", gt.column_names)
truth_df = pd.DataFrame({
    "ID": gt["ID"],
    "Reference_phn":  gt["Reference_phn"],
    "Annotation_phn": gt["Annotation_phn"],
})
truth_df.to_csv("truth.csv", index=False)
print("wrote truth.csv:", truth_df.shape)
print(truth_df.head(2).to_string())

In [ ]:
# ---- Run the OFFICIAL scorer ------------------------------------------------
# Step 1: align_data.py writes ref_our_detail / ref_human_detail / human_our_detail
# Step 2: ins_del_cor_sub_analysis.py reads those and prints Precision/Recall/F1.
import subprocess, sys
EVAL = "interspeech_IqraEval/mdd_eval"

r1 = subprocess.run(
    [sys.executable, f"{EVAL}/align_data.py",
     "--truth-file", "truth.csv", "--pred-file", "pred.csv",
     "--output-dir", "aligned"],
    capture_output=True, text=True)
print("align stdout:\n", r1.stdout[-2000:])
print("align stderr:\n", r1.stderr[-2000:])
!ls -la aligned

In [ ]:
# Their script does root + "ref_human_detail" with no separator, so the folder
# arg MUST end in a slash: "aligned/" not "aligned".
r2 = subprocess.run(
    [sys.executable, f"{EVAL}/ins_del_cor_sub_analysis.py", "--align-folder", "aligned/"],
    capture_output=True, text=True)
print("=========== OFFICIAL IqraEval SCORE ===========")
print(r2.stdout)
if r2.stderr:
    print("stderr:\n", r2.stderr[-3000:])

### Reading the result
The printed **f1-score** is your real, leaderboard-comparable number. Field refs:
baseline F1 ~0.44 (IQRA'26) / 0.30-0.44 (2025) | AraS2P winner 0.47 | best 0.72.

**If the args don't match** (the scripts may use `--out-file` or read `aligned/`
by default), check the printed stderr and adjust the two subprocess calls — open
`interspeech_IqraEval/mdd_eval/align_data.py` and `ins_del_cor_sub_analysis.py`
to confirm exact flag names. The first cell's `!ls` shows the scripts present.